# Fine-tune Llama 3.2 1B for SOP Generation (Colab)

This notebook trains a **LoRA adapter** for `meta-llama/Llama-3.2-1B-Instruct` on 400 training SOPs and validates on 100 SOPs.

Key safeguards:
- explicit Colab paths and file checks;
- dataset schema validation;
- assistant-only loss with boundary checks;
- sequence-length and truncation reporting;
- aligned evaluation/checkpoint steps, best-model loading, and early stopping;
- adapter reload test and ZIP integrity check before download.

## 1. Install dependencies

Run this cell once, then use **Runtime → Restart session** and continue from the next cell.

In [ ]:
!pip install -q transformers==4.45.0 accelerate==0.34.0 peft==0.13.0 datasets==2.21.0
print("Dependencies installed. Restart the session before continuing.")

## 2. Check the runtime and inputs

Upload `train.jsonl` and `val.jsonl` with the Files panel before running this cell.

In [ ]:
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, PeftConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
TRAIN_PATH = Path("/content/train.jsonl")
VAL_PATH = Path("/content/val.jsonl")
OUTPUT_DIR = Path("/content/sop_training")
ADAPTER_DIR = Path("/content/sop_lora_adapter")
MAX_LENGTH = 2048

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), "Enable a T4 GPU under Runtime → Change runtime type."
assert TRAIN_PATH.is_file(), f"Missing {TRAIN_PATH}. Upload train.jsonl in the Files panel."
assert VAL_PATH.is_file(), f"Missing {VAL_PATH}. Upload val.jsonl in the Files panel."

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Training data:", TRAIN_PATH)
print("Validation data:", VAL_PATH)

## 3. Validate and load the dataset

In [ ]:
ALLOWED_ROLES = {"system", "user", "assistant"}

def validate_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {path}:{line_number}: {error}") from error

            messages = record.get("messages")
            if not isinstance(messages, list) or not messages:
                raise ValueError(f"Missing messages list at {path}:{line_number}")

            roles = []
            for message in messages:
                if not isinstance(message, dict):
                    raise ValueError(f"Invalid message at {path}:{line_number}")
                role = message.get("role")
                content = message.get("content")
                if role not in ALLOWED_ROLES:
                    raise ValueError(f"Invalid role {role!r} at {path}:{line_number}")
                if not isinstance(content, str) or not content.strip():
                    raise ValueError(f"Empty content at {path}:{line_number}")
                roles.append(role)

            if roles[-1] != "assistant" or roles.count("assistant") != 1:
                raise ValueError(
                    f"Expected exactly one final assistant message at {path}:{line_number}; got {roles}"
                )
            records.append(record)
    return records

train_records = validate_jsonl(TRAIN_PATH)
val_records = validate_jsonl(VAL_PATH)

assert len(train_records) == 400, f"Expected 400 training records, got {len(train_records)}"
assert len(val_records) == 100, f"Expected 100 validation records, got {len(val_records)}"

train_dataset = load_dataset("json", data_files=str(TRAIN_PATH), split="train")
val_dataset = load_dataset("json", data_files=str(VAL_PATH), split="train")

print("Validated training records:", len(train_dataset))
print("Validated validation records:", len(val_dataset))
print("Columns:", train_dataset.column_names)

## 4. Log in to Hugging Face

The account must have accepted the Llama 3.2 license.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Load Llama 3.2 1B and attach LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Build assistant-only labels and inspect sequence lengths

The cell verifies that the prompt tokens are an exact prefix of the full conversation. Examples longer than `MAX_LENGTH` are rejected instead of silently truncating the SOP.

In [ ]:
def tokenize_example(example):
    messages = example["messages"]
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True,
    )
    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt/full token boundary mismatch")

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    supervised_tokens = sum(label != -100 for label in labels)
    valid = len(full_ids) <= MAX_LENGTH and supervised_tokens > 0

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "sequence_length": len(full_ids),
        "supervised_tokens": supervised_tokens,
        "valid": valid,
    }

raw_train_columns = train_dataset.column_names
raw_val_columns = val_dataset.column_names
tokenized_train = train_dataset.map(tokenize_example, remove_columns=raw_train_columns)
tokenized_val = val_dataset.map(tokenize_example, remove_columns=raw_val_columns)

def report_lengths(name, dataset):
    lengths = np.array(dataset["sequence_length"])
    supervised = np.array(dataset["supervised_tokens"])
    print(
        f"{name}: count={len(dataset)}, max={lengths.max()}, "
        f"p95={np.percentile(lengths, 95):.0f}, over_limit={(lengths > MAX_LENGTH).sum()}, "
        f"zero_supervision={(supervised == 0).sum()}"
    )

report_lengths("Train", tokenized_train)
report_lengths("Validation", tokenized_val)

invalid_train = sum(not value for value in tokenized_train["valid"])
invalid_val = sum(not value for value in tokenized_val["valid"])
assert invalid_train == 0 and invalid_val == 0, (
    f"Found invalid/overlength examples: train={invalid_train}, validation={invalid_val}. "
    "Inspect them before training instead of silently truncating."
)

keep_columns = {"input_ids", "attention_mask", "labels"}
tokenized_train = tokenized_train.remove_columns(
    [name for name in tokenized_train.column_names if name not in keep_columns]
)
tokenized_val = tokenized_val.remove_columns(
    [name for name in tokenized_val.column_names if name not in keep_columns]
)
print("Assistant-only tokenization checks passed.")

## 7. Train with best-checkpoint loading and early stopping

In [ ]:
class AssistantOnlyCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        input_features = [
            {"input_ids": item["input_ids"], "attention_mask": item["attention_mask"]}
            for item in features
        ]
        batch = self.tokenizer.pad(input_features, padding=True, return_tensors="pt")
        max_length = batch["input_ids"].shape[1]
        padded_labels = []
        for item in features:
            labels = list(item["labels"])
            labels.extend([-100] * (max_length - len(labels)))
            padded_labels.append(labels)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=AssistantOnlyCollator(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
eval_metrics = trainer.evaluate()
print("Training metrics:", train_result.metrics)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Final evaluation:", eval_metrics)

## 8. Save, verify, ZIP, and download the adapter

In [ ]:
if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run_metadata = {
    "base_model": BASE_MODEL,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "training_examples": len(tokenized_train),
    "validation_examples": len(tokenized_val),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "eval_metrics": eval_metrics,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
}
with (ADAPTER_DIR / "run_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2, default=str)

required_files = [
    ADAPTER_DIR / "adapter_config.json",
    ADAPTER_DIR / "adapter_model.safetensors",
    ADAPTER_DIR / "tokenizer_config.json",
    ADAPTER_DIR / "run_metadata.json",
]
missing = [str(path) for path in required_files if not path.is_file() or path.stat().st_size == 0]
assert not missing, f"Missing or empty adapter files: {missing}"

# Verify that PEFT can reload the saved adapter configuration.
reloaded_config = PeftConfig.from_pretrained(ADAPTER_DIR)
assert reloaded_config.base_model_name_or_path == BASE_MODEL

zip_path = Path(shutil.make_archive("/content/sop_lora_adapter", "zip", root_dir="/content", base_dir="sop_lora_adapter"))
assert zip_path.is_file() and zip_path.stat().st_size > 0
shutil.unpack_archive(zip_path, "/content/adapter_zip_test")
assert Path("/content/adapter_zip_test/sop_lora_adapter/adapter_config.json").is_file()

print("Verified adapter:", ADAPTER_DIR)
print("Verified ZIP:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")

from google.colab import files
files.download(str(zip_path))

## Important evaluation note

The supplied validation split may contain augmented variants related to training examples. Treat its loss and later ROUGE/BERTScore results as preliminary until the data-preparation pipeline provides a leakage-resistant grouped split and an independent test set.